In [6]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    get_linear_schedule_with_warmup,
)


candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "processed" / "incidents_train.csv").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed/incidents_train.csv"
    )

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "incidents_train.csv"
VALIDATION_DATA_PATH = PROCESSED_DATA_DIR / "incidents_validation.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "incidents_test.csv"
LABEL_MAPPING_PATH = PROCESSED_DATA_DIR / "department_label_mapping.csv"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "train" / "artifacts" / "distilbert_incident_classifier"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_DATA_PATH)
validation_df = pd.read_csv(VALIDATION_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)
label_mapping_df = pd.read_csv(LABEL_MAPPING_PATH).sort_values("department_label")

label_names = label_mapping_df["assigned_department"].tolist()
label_to_name = dict(zip(label_mapping_df["department_label"], label_mapping_df["assigned_department"]))
num_labels = len(label_mapping_df)

print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(validation_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Number of labels: {num_labels}")

train_df[["feature_concatanation", "department_label"]].head()

Train rows: 2520
Validation rows: 540
Test rows: 540
Number of labels: 12


,feature_concatanation,department_label
0,Ward Sister - Medical Please send porter team ...,11
1,Pharmacy Manager Antibiotic bulk store damp; h...,7
2,"Labour Room Midwife Between cases, blood splas...",4
3,Pharmacy Intern - Inpatient Glucometer strips ...,10
4,"Head, Physiotherapy TENS units due for annual ...",0


In [7]:
TEXT_COLUMN = "feature_concatanation"
LABEL_COLUMN = "department_label"
MODEL_NAME = "distilbert-base-uncased"
BASE_MODEL_SOURCE = MODEL_NAME
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
RANDOM_SEED = 42


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_tokenizer(model_source):
    try:
        return DistilBertTokenizerFast.from_pretrained(model_source)
    except OSError as exc:
        raise RuntimeError(
            f"Tokenizer files for '{model_source}' are not available locally. "
            "Download the model once with internet access or point BASE_MODEL_SOURCE to a local model directory."
        ) from exc


tokenizer = load_tokenizer(BASE_MODEL_SOURCE)


class IncidentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, text_column, label_column, max_length):
        texts = dataframe[text_column].fillna("").astype(str).tolist()
        labels = dataframe[label_column].astype(int).tolist()

        self.labels = torch.tensor(labels, dtype=torch.long)
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_attention_mask=True,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx], dtype=torch.long)
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[idx]
        return item


train_dataset = IncidentDataset(train_df, tokenizer, TEXT_COLUMN, LABEL_COLUMN, MAX_LENGTH)
validation_dataset = IncidentDataset(validation_df, tokenizer, TEXT_COLUMN, LABEL_COLUMN, MAX_LENGTH)
test_dataset = IncidentDataset(test_df, tokenizer, TEXT_COLUMN, LABEL_COLUMN, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Using device: {device}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(validation_loader)}")
print(f"Test batches: {len(test_loader)}")

Using device: cpu
Training batches: 158
Validation batches: 34
Test batches: 34


In [8]:
def load_base_model(model_source):
    try:
        return DistilBertForSequenceClassification.from_pretrained(
            model_source,
            num_labels=num_labels,
            id2label={idx: name for idx, name in enumerate(label_names)},
            label2id={name: idx for idx, name in enumerate(label_names)},
        )
    except OSError as exc:
        raise RuntimeError(
            f"Base model '{model_source}' is not available locally. "
            "Download it once with internet access or point BASE_MODEL_SOURCE to a local pretrained DistilBERT directory."
        ) from exc


model = load_base_model(BASE_MODEL_SOURCE)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_training_steps = len(train_loader) * EPOCHS
warmup_steps = math.ceil(total_training_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)


def move_batch_to_device(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def compute_classification_metrics(predictions, labels, num_labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)

    accuracy = float((predictions == labels).mean()) if len(labels) else 0.0
    f1_scores = []

    for label_idx in range(num_labels):
        true_positive = np.sum((predictions == label_idx) & (labels == label_idx))
        false_positive = np.sum((predictions == label_idx) & (labels != label_idx))
        false_negative = np.sum((predictions != label_idx) & (labels == label_idx))

        precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else 0.0
        recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        f1_scores.append(f1)

    macro_f1 = float(np.mean(f1_scores)) if f1_scores else 0.0
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
    }


def evaluate_model(model, data_loader, device, num_labels):
    model.eval()
    losses = []
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            batch = move_batch_to_device(batch, device)
            outputs = model(**batch)
            losses.append(outputs.loss.item())

            predictions = torch.argmax(outputs.logits, dim=1)
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())

    metrics = compute_classification_metrics(all_predictions, all_labels, num_labels)
    metrics["loss"] = float(np.mean(losses)) if losses else 0.0
    return metrics, np.asarray(all_predictions), np.asarray(all_labels)


sample_batch = next(iter(train_loader))
sample_batch = move_batch_to_device(sample_batch, device)
sample_outputs = model(**sample_batch)
print(f"Smoke-test loss: {sample_outputs.loss.item():.4f}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1854.60it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Smoke-test loss: 2.5013


In [9]:
history = []
best_validation_loss = float("inf")
best_model_path = MODEL_OUTPUT_DIR / "best_model_state.pt"

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for batch in progress_bar:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        progress_bar.set_postfix(train_loss=loss.item())

    average_train_loss = running_loss / max(len(train_loader), 1)
    validation_metrics, _, _ = evaluate_model(model, validation_loader, device, num_labels)

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": average_train_loss,
        "validation_loss": validation_metrics["loss"],
        "validation_accuracy": validation_metrics["accuracy"],
        "validation_macro_f1": validation_metrics["macro_f1"],
    }
    history.append(epoch_result)
    print(epoch_result)

    if validation_metrics["loss"] < best_validation_loss:
        best_validation_loss = validation_metrics["loss"]
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
history_df.to_csv(MODEL_OUTPUT_DIR / "training_history.csv", index=False)
history_df

Epoch 1/5: 100%|██████████| 158/158 [07:40<00:00,  2.91s/it, train_loss=1.4] 


{'epoch': 1, 'train_loss': 2.0621483484400978, 'validation_loss': 1.1517602345522713, 'validation_accuracy': 0.7962962962962963, 'validation_macro_f1': 0.7992306411400388}


Epoch 2/5: 100%|██████████| 158/158 [08:06<00:00,  3.08s/it, train_loss=0.261]


{'epoch': 2, 'train_loss': 0.6790224353346643, 'validation_loss': 0.42979781285804863, 'validation_accuracy': 0.912962962962963, 'validation_macro_f1': 0.9113299099564109}


Epoch 3/5: 100%|██████████| 158/158 [07:39<00:00,  2.91s/it, train_loss=0.118] 


...

Epoch 4/5: 100%|██████████| 158/158 [07:29<00:00,  2.85s/it, train_loss=0.0533]


{'epoch': 4, 'train_loss': 0.10340711719627622, 'validation_loss': 0.24554732858258135, 'validation_accuracy': 0.9388888888888889, 'validation_macro_f1': 0.9366439127896141}


Epoch 5/5: 100%|██████████| 158/158 [07:34<00:00,  2.88s/it, train_loss=0.0334]


{'epoch': 5, 'train_loss': 0.06304917108456168, 'validation_loss': 0.24004728056709557, 'validation_accuracy': 0.9388888888888889, 'validation_macro_f1': 0.9365018703787614}


,epoch,train_loss,validation_loss,validation_accuracy,validation_macro_f1
0,1,2.062148,1.151760,0.796296,0.799231
1,2,0.679022,0.429798,0.912963,0.911330
2,3,0.235360,0.279796,0.924074,0.921993
3,4,0.103407,0.245547,0.938889,0.936644
4,5,0.063049,0.240047,0.938889,0.936502


In [10]:
best_model = load_base_model(BASE_MODEL_SOURCE)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.to(device)

best_model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

with open(MODEL_OUTPUT_DIR / "label_mapping.json", "w", encoding="utf-8") as file:
    json.dump(label_to_name, file, indent=2)

print(f"Saved fine-tuned model to: {MODEL_OUTPUT_DIR}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4047.97it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.78it/s]

Saved fine-tuned model to: /home/lakshan/ssp/IMS/train/artifacts/distilbert_incident_classifier


In [11]:
saved_model = DistilBertForSequenceClassification.from_pretrained(MODEL_OUTPUT_DIR)
saved_model.to(device)

test_metrics, test_predictions, test_labels = evaluate_model(saved_model, test_loader, device, num_labels)
ordered_labels = sorted(label_to_name)
target_names = [label_to_name[label] for label in ordered_labels]

print(test_metrics)
print("\nTest classification report:\n")
print(
    classification_report(
        test_labels,
        test_predictions,
        labels=ordered_labels,
        target_names=target_names,
        zero_division=0,
    )
)

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(test_labels, test_predictions, labels=ordered_labels),
    index=[f"actual_{label_to_name[label]}" for label in ordered_labels],
    columns=[f"pred_{label_to_name[label]}" for label in ordered_labels],
)

print("\nConfusion matrix:\n")
print(confusion_matrix_df)

test_results_df = test_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_results_df["predicted_label"] = test_predictions
test_results_df["actual_department"] = test_results_df[LABEL_COLUMN].map(label_to_name)
test_results_df["predicted_department"] = test_results_df["predicted_label"].map(label_to_name)
test_results_df["is_correct"] = test_results_df[LABEL_COLUMN] == test_results_df["predicted_label"]

test_results_df.to_csv(MODEL_OUTPUT_DIR / "test_predictions.csv", index=False)
test_results_df.head(10)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5277.58it/s]


{'accuracy': 0.9629629629629629, 'macro_f1': 0.9643288123304851, 'loss': 0.14290968624546246}

Test classification report:

                              precision    recall  f1-score   support

      Biomedical Engineering       0.98      0.95      0.96        42
   Dietary and Food Services       0.96      0.98      0.97        45
         Facility Management       0.96      0.98      0.97        48
         Finance and Account       0.96      0.96      0.96        51
     Housekeeping Department       0.96      0.96      0.96        46
             Human Resources       0.96      0.98      0.97        45
                          IT       0.98      0.98      0.98        45
Quality Assurance department       1.00      0.94      0.97        32
                   Reception       1.00      0.96      0.98        45
                    Security       1.00      0.97      0.99        40
           Supply Department       0.91      0.91      0.91        55
                   Transport       

,feature_concatanation,department_label,predicted_label,actual_department,predicted_department,is_correct
0,Dialysis Unit Sister Post-dialysis wheelchair ...,11,11,Transport,Transport,True
1,Ward Block Security Officer Digital board at R...,8,8,Reception,Reception,True
2,Infection Control Nurse Dry store feels damp a...,1,1,Dietary and Food Services,Dietary and Food Services,True
3,Theatre Sister Main OT AHU not meeting tempera...,2,2,Facility Management,Facility Management,True
4,Ward Nurse - NICU Handheld vitals tablets drop...,6,6,IT,IT,True
5,Consultant Physician eChannelling commission d...,3,3,Finance and Account,Finance and Account,True
6,Pharmacist - OPD Dispensary Inventory quantiti...,6,6,IT,IT,True
7,ICU Medical Officer Urgent noradrenaline and h...,11,11,Transport,Transport,True
8,ETU Nursing Sister Ambulance 05 contaminated w...,11,11,Transport,Transport,True
9,Ward Sister - Female Medical Power flicker wit...,7,7,Quality Assurance department,Quality Assurance department,True
